In [2]:
import pandas as pd

In [3]:
df = pd.read_parquet("../data/raw/sdwpf_2001_2112_full.parquet")  # adjust filename to what you actually downloaded
print(df.shape)
print(df.columns.tolist())
print(df['TurbID'].nunique(), "turbines")
df.head()

(11361190, 18)
['TurbID', 'Tmstamp', 'Wspd', 'Wdir', 'Etmp', 'Itmp', 'Ndir', 'Pab1', 'Pab2', 'Pab3', 'Prtv', 'T2m', 'Sp', 'RelH', 'Wspd_w', 'Wdir_w', 'Tp', 'Patv']
134 turbines


,TurbID,Tmstamp,Wspd,Wdir,Etmp,Itmp,Ndir,Pab1,Pab2,Pab3,Prtv,T2m,Sp,RelH,Wspd_w,Wdir_w,Tp,Patv
0,57,2020-01-03 04:25:00,7.935,0.055,2.185,23.415,236.59,-0.840,-0.840,-0.845,29.680,-12.312683,85893.515625,0.529154,5.191925,52.389540,0.0,976.925
1,57,2020-01-03 04:55:00,6.425,-1.860,2.945,23.550,236.59,-0.610,-0.605,-0.610,14.480,-12.312683,85893.515625,0.529154,5.191925,52.389540,0.0,610.830
2,57,2020-01-03 05:25:00,5.945,-1.635,3.720,23.350,236.59,-0.435,-0.435,-0.435,15.270,-11.709015,85836.429688,0.516070,5.563238,56.420248,0.0,483.850
3,57,2020-01-03 05:55:00,6.290,1.220,4.385,23.285,236.59,-0.510,-0.510,-0.510,19.030,-11.709015,85836.429688,0.516070,5.563238,56.420248,0.0,536.970
4,57,2020-01-03 06:25:00,5.650,3.790,4.945,23.265,243.84,-0.355,-0.355,-0.355,9.795,-10.475830,85820.765625,0.515158,6.000677,55.780818,0.0,431.700


In [3]:
df_csv = pd.read_csv("../data/raw/sdwpf_2001_2112_full.csv")

print(df.shape == df_csv.shape)
print(sorted(df.columns) == sorted(df_csv.columns))
print(df.dtypes)
print(df_csv.dtypes)

True
True
TurbID              int64
Tmstamp    datetime64[ns]
Wspd              float64
Wdir              float64
Etmp              float64
Itmp              float64
Ndir              float64
Pab1              float64
Pab2              float64
Pab3              float64
Prtv              float64
T2m               float32
Sp                float32
RelH              float32
Wspd_w            float64
Wdir_w            float64
Tp                float32
Patv              float64
dtype: object
TurbID       int64
Tmstamp        str
Wspd       float64
Wdir       float64
Etmp       float64
Itmp       float64
Ndir       float64
Pab1       float64
Pab2       float64
Pab3       float64
Prtv       float64
T2m        float64
Sp         float64
RelH       float64
Wspd_w     float64
Wdir_w     float64
Tp         float64
Patv       float64
dtype: object


In [4]:
df = df.sort_values(["TurbID", "Tmstamp"])
df["delta"] = df.groupby("TurbID")["Tmstamp"].diff()

# Distribution of gaps across the whole dataset
print(df["delta"].value_counts().sort_index())

delta
0 days 00:10:00     7043040
0 days 00:15:00     4317748
0 days 08:15:00         134
30 days 00:15:00        134
Name: count, dtype: int64


In [5]:
# Does the 15-min gap correlate with specific turbines?
df["delta"].groupby(df["TurbID"]).value_counts().unstack().head(20)

# Or with specific date ranges?
fifteen_min_rows = df[df["delta"] == pd.Timedelta(minutes=15)]
print(fifteen_min_rows["Tmstamp"].dt.date.value_counts().sort_index())

Tmstamp
2020-01-01    12730
2020-01-02    12864
2020-01-03    12864
2020-01-04    12864
2020-01-05    12864
              ...  
2020-12-27    12864
2020-12-28    12864
2020-12-29    12864
2020-12-30    12864
2020-12-31     8576
Name: count, Length: 336, dtype: int64


In [6]:
big_gaps = df[df["delta"].isin([pd.Timedelta(hours=8, minutes=15), pd.Timedelta(days=30, minutes=15)])]
print(big_gaps[["TurbID", "Tmstamp", "delta"]].sort_values("Tmstamp"))

          TurbID             Tmstamp            delta
3081760        1 2020-05-01 00:10:00 30 days 00:15:00
2485616       53 2020-05-01 00:10:00 30 days 00:15:00
2501728       54 2020-05-01 00:10:00 30 days 00:15:00
3694016      107 2020-05-01 00:10:00 30 days 00:15:00
2517840       55 2020-05-01 00:10:00 30 days 00:15:00
...          ...                 ...              ...
4580821       62 2021-01-01 00:10:00  0 days 08:15:00
4633382       63 2021-01-01 00:10:00  0 days 08:15:00
4685943       64 2021-01-01 00:10:00  0 days 08:15:00
6105090       44 2021-01-01 00:10:00  0 days 08:15:00
10730458     134 2021-01-01 00:10:00  0 days 08:15:00

[268 rows x 3 columns]
